# 11 — The final model, and predicting a fight

After the test look, the model is refitted on **every decided fight from 2014 to the last
event in the data**, with the same frozen recipe. This is the model that predicts upcoming
fights. Its own score cannot be measured (no unseen fights are left); the test look in
`10_test_evaluation.ipynb` is the honest estimate of how well this recipe predicts.

The calibration stretch is the one fitted in the test look: refitting it here would need
out-of-fold predictions on the test years, which the tuning guard refuses on purpose.

`modeling.final.predict_upcoming` adds a fight to the history as a fight with no statistics
and no result, then reads its features from the normal pipeline: the same code that built
every past row, so nothing about a future fight is computed differently.

In [1]:
import json
import pandas as pd

from ufc_predictor.config import RAW_DIR
from ufc_predictor.data.io import load_fighters, load_processed
from ufc_predictor.features.elo import load_elo_params, load_elo_weights
from ufc_predictor.modeling.final import fit_final, load_bundle, predict_upcoming, save_bundle
from ufc_predictor.modeling.tune import load_params

fights, fighters = load_processed("fights"), load_fighters()
source = json.loads((RAW_DIR / "source" / "SOURCE.json").read_text())
until = pd.Timestamp(source["last_event"]) + pd.Timedelta(days=1)
bundle = fit_final(fights, fighters, until=until, params=load_params(),
                   elo_k=load_elo_params(), elo_weights=load_elo_weights(), calibrate=False)
bundle["stretch"] = json.load(open("reviews/test_result_2026-10-17.json"))["stretch"]
bundle["data_version"] = source["version"]
path = save_bundle(bundle)
print(f"trained on {bundle['training_fights']:,} fights to {source['last_event']} (data v{source['version']}); "
      f"stretch {bundle['stretch']:.3f}; saved to {path.name}")

trained on 6,276 fights to 2026-10-10 (data v20); stretch 1.235; saved to final_model.joblib


## Predicting a fight

An example, **not a scheduled bout**: rematches of two fights from the last card in the data,
as if held on 2026-12-05. Fighters are named as in the fighter table (or by id); the
probability is for the fighter listed first.

In [2]:
bouts = [
    {"fighter_a": "Brendan Allen", "fighter_b": "Christian Leroy Duncan", "date": "2026-12-05", "weight_class": "Middleweight Bout"},
    {"fighter_a": "Andre Fili", "fighter_b": "Kai Kamaka III", "date": "2026-12-05", "weight_class": "Featherweight Bout"},
]
predict_upcoming(bouts, load_bundle())

,date,fighter,opponent,p_win,fighter_debut,opponent_debut
0,2026-12-05,Brendan Allen,Christian Leroy Duncan,0.542798,False,False
1,2026-12-05,Andre Fili,Kai Kamaka III,0.563904,False,False
